# Chapter 03: Isometry in the Euclidean Plane

_Source span: Coxeter, Introduction to Geometry, Chapter 3, printed pages 39-49 (PDF pages 57-67)._

This notebook treats a plane isometry as a concrete map

\[
(x,y,1)^T \longmapsto
\begin{bmatrix} A & t \\ 0 & 1 \end{bmatrix}(x,y,1)^T,
\]

where the linear part `A` is orthogonal. The chapter goal is to make Coxeter's catalog inspectable: direct isometries preserve sense and are rotations or translations; opposite isometries reverse sense and are reflections or glide reflections. We will test those statements with triangles, mirrors, half-turns, midpoints, and strip patterns.


## Computational Translation Guide

| Book idea | Computational representation | What to inspect |
| --- | --- | --- |
| Sense of a directed triangle | Signed area of three image points | The sign changes exactly when `det(A) = -1`. |
| Direct or opposite isometry | Orthogonal matrix with determinant `+1` or `-1` | Direct maps keep orientation; opposite maps reverse it. |
| Reflection in a mirror | Matrix `A = 2uu^T - I` for the mirror direction `u` | Fixed set is a line; every product changes determinant by multiplication. |
| Half-turn about a point | Rotation by `pi`: `x -> 2c - x` | Two half-turns compose to a translation. |
| Translation | `A = I` with nonzero vector `t` | No finite fixed point; translations commute. |
| Glide reflection | Reflection in an axis followed by translation along that axis | Opposite, no fixed point, but segment midpoints lie on the axis. |
| Strip pattern | A motif repeated by a small generating set | Which generators are translations, reflections, glides, or half-turns. |


## Compact Visual Storyboard Implemented

| Storyboard item | Artifact | Learner inspection target | Validation |
| --- | --- | --- | --- |
| Matrix classifier for direct/opposite isometries | `figures/transformation_matrix_classifier.svg`, `tables/transformation_matrix_classifier.csv` | Same triangle under six isometries; determinant and fixed-set type beside each map | Pairwise distances preserved; signed-area ratio equals determinant |
| Product of reflections and half-turns | `figures/reflection_line_product_lab.svg`, `checks/reflection_product_checks.json` | Intersecting mirrors rotate, parallel mirrors translate, half-turn pairs translate | Numeric residuals and exact SymPy identities |
| Glide reflection strip motif | `figures/glide_reflection_strip_motif.svg`, `html/glide_reflection_strip_motif.html` | Alternating motifs and midpoint traces along the glide axis | `det=-1`, no fixed point, `G^2` is a translation, midpoints land on the axis |
| Hjelmslev midpoint scaffold | `figures/hjelmslev_midpoint_scaffold.svg` | Midpoints from a line to its image are collinear, or all coincide in a half-turn | Rank and spread checks on midpoint sets |
| Seven strip-pattern generator types | `figures/strip_pattern_gallery.svg`, `tables/strip_pattern_summary.csv` | Which visible symmetry operations generate each strip row | Seven rows, generator determinant checks, period sanity |


In [1]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display
from matplotlib.patches import Arc, Circle, Polygon


def find_book_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").exists() and candidate.name == "Introduction-to-Geometry":
            return candidate
        nested = candidate / "Introduction-to-Geometry"
        if (nested / "AGENTS.md").exists():
            return nested
    raise RuntimeError("Could not locate Introduction-to-Geometry root")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (  # noqa: E402
    artifact_path,
    assert_artifacts,
    display_artifact,
    ensure_chapter_artifact_dirs,
    write_csv,
    write_json,
)

CHAPTER_NO = 3
ARTIFACT_DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)

# Remove the known generic scaffold artifacts for this chapter. The names are course scaffolding,
# not source-specific visuals, and the audit script treats them as stale.
for stale in [
    artifact_path(CHAPTER_NO, "figures", "concept_configuration.svg", BOOK_ROOT),
    artifact_path(CHAPTER_NO, "figures", "parameter_experiment.svg", BOOK_ROOT),
]:
    if stale.exists():
        stale.unlink()

PATHS = {
    "classifier_figure": artifact_path(CHAPTER_NO, "figures", "transformation_matrix_classifier.svg", BOOK_ROOT),
    "classifier_table": artifact_path(CHAPTER_NO, "tables", "transformation_matrix_classifier.csv", BOOK_ROOT),
    "orientation_checks": artifact_path(CHAPTER_NO, "checks", "orientation_determinant_checks.json", BOOK_ROOT),
    "reflection_lab_figure": artifact_path(CHAPTER_NO, "figures", "reflection_line_product_lab.svg", BOOK_ROOT),
    "reflection_checks": artifact_path(CHAPTER_NO, "checks", "reflection_product_checks.json", BOOK_ROOT),
    "group_table": artifact_path(CHAPTER_NO, "tables", "group_composition_table.csv", BOOK_ROOT),
    "glide_svg": artifact_path(CHAPTER_NO, "figures", "glide_reflection_strip_motif.svg", BOOK_ROOT),
    "glide_html": artifact_path(CHAPTER_NO, "html", "glide_reflection_strip_motif.html", BOOK_ROOT),
    "glide_checks": artifact_path(CHAPTER_NO, "checks", "glide_reflection_checks.json", BOOK_ROOT),
    "hjelmslev_figure": artifact_path(CHAPTER_NO, "figures", "hjelmslev_midpoint_scaffold.svg", BOOK_ROOT),
    "hjelmslev_checks": artifact_path(CHAPTER_NO, "checks", "hjelmslev_midpoint_checks.json", BOOK_ROOT),
    "strip_gallery": artifact_path(CHAPTER_NO, "figures", "strip_pattern_gallery.svg", BOOK_ROOT),
    "strip_table": artifact_path(CHAPTER_NO, "tables", "strip_pattern_summary.csv", BOOK_ROOT),
    "strip_checks": artifact_path(CHAPTER_NO, "checks", "strip_pattern_checks.json", BOOK_ROOT),
    "sample_data": artifact_path(CHAPTER_NO, "data", "isometry_sample_matrices.json", BOOK_ROOT),
    "manifest": artifact_path(CHAPTER_NO, "tables", "artifact_manifest.csv", BOOK_ROOT),
    "visual_summary": artifact_path(CHAPTER_NO, "checks", "visual_summary.json", BOOK_ROOT),
    "sanity_summary": artifact_path(CHAPTER_NO, "checks", "final_sanity_summary.json", BOOK_ROOT),
}

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 140,
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})


def homogeneous(A: np.ndarray, t=(0.0, 0.0)) -> np.ndarray:
    M = np.eye(3)
    M[:2, :2] = np.asarray(A, dtype=float)
    M[:2, 2] = np.asarray(t, dtype=float)
    return M


def translation(v) -> np.ndarray:
    return homogeneous(np.eye(2), v)


def rotation(theta: float, center=(0.0, 0.0)) -> np.ndarray:
    c = np.asarray(center, dtype=float)
    A = np.array([[math.cos(theta), -math.sin(theta)], [math.sin(theta), math.cos(theta)]])
    return homogeneous(A, c - A @ c)


def reflection(angle: float, point=(0.0, 0.0)) -> np.ndarray:
    p = np.asarray(point, dtype=float)
    u = np.array([math.cos(angle), math.sin(angle)])
    A = 2 * np.outer(u, u) - np.eye(2)
    return homogeneous(A, p - A @ p)


def half_turn(center=(0.0, 0.0)) -> np.ndarray:
    return rotation(math.pi, center)


def glide_reflection(axis_angle=0.0, point=(0.0, 0.0), distance=1.0) -> np.ndarray:
    direction = np.array([math.cos(axis_angle), math.sin(axis_angle)])
    return translation(distance * direction) @ reflection(axis_angle, point)


def reflect_horizontal(y0: float) -> np.ndarray:
    return homogeneous(np.array([[1.0, 0.0], [0.0, -1.0]]), (0.0, 2.0 * y0))


def apply_transform(M: np.ndarray, pts: np.ndarray) -> np.ndarray:
    pts = np.asarray(pts, dtype=float)
    pts_h = np.c_[pts, np.ones(len(pts))]
    mapped = (M @ pts_h.T).T
    return mapped[:, :2] / mapped[:, 2:3]


def signed_area2(triangle: np.ndarray) -> float:
    a, b, c = np.asarray(triangle, dtype=float)
    u = b - a
    v = c - a
    return float(u[0] * v[1] - u[1] * v[0])


def pairwise_distances(pts: np.ndarray) -> np.ndarray:
    pts = np.asarray(pts, dtype=float)
    delta = pts[:, None, :] - pts[None, :, :]
    return np.sqrt((delta**2).sum(axis=-1))


def fixed_set_type(M: np.ndarray, tol=1e-9) -> str:
    B = M[:2, :2] - np.eye(2)
    rhs = -M[:2, 2]
    rank_B = np.linalg.matrix_rank(B, tol)
    rank_aug = np.linalg.matrix_rank(np.c_[B, rhs], tol)
    if rank_aug > rank_B:
        return "none"
    return {0: "whole plane", 1: "line", 2: "point"}[rank_B]


def determinant_sense(M: np.ndarray) -> str:
    return "direct" if np.linalg.det(M[:2, :2]) > 0 else "opposite"


def matrix_power(M: np.ndarray, n: int) -> np.ndarray:
    if n >= 0:
        return np.linalg.matrix_power(M, n)
    return np.linalg.matrix_power(np.linalg.inv(M), -n)


def draw_segment(ax, p, q, color="#455a64", linewidth=1.3, linestyle="-"):
    ax.plot([p[0], q[0]], [p[1], q[1]], color=color, lw=linewidth, ls=linestyle)


def draw_polygon(ax, pts, *, edge="#263238", face="#b3e5fc", alpha=0.85, label=None, lw=1.4):
    patch = Polygon(pts, closed=True, facecolor=face, edgecolor=edge, lw=lw, alpha=alpha, label=label)
    ax.add_patch(patch)
    closed = np.vstack([pts, pts[0]])
    ax.plot(closed[:, 0], closed[:, 1], color=edge, lw=lw)


def set_equal_window(ax, pts, margin=0.45):
    pts = np.asarray(pts, dtype=float)
    lo = pts.min(axis=0) - margin
    hi = pts.max(axis=0) + margin
    span = max(hi[0] - lo[0], hi[1] - lo[1])
    cx, cy = (lo + hi) / 2
    ax.set_xlim(cx - span / 2, cx + span / 2)
    ax.set_ylim(cy - span / 2, cy + span / 2)
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0, color="#cfd8dc", lw=0.8)
    ax.axvline(0, color="#cfd8dc", lw=0.8)
    ax.grid(True, color="#eceff1", lw=0.6)


def book_rel(path: Path) -> str:
    return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()


display(Markdown(f"Artifact root: `{book_rel(BOOK_ROOT / 'artifacts' / 'chapter-03')}`"))


Artifact root: `artifacts/chapter-03`

## 1. Direct and Opposite Isometries as Matrix Tests

A single triangle is enough to expose the sense of an isometry. If the signed area of the image triangle has the same sign, the isometry is direct. If the sign is reversed, it is opposite. The determinant of the linear part predicts exactly the same sign, while the fixed-set computation separates translations from rotations and glide reflections from ordinary reflections.


In [2]:
TRIANGLE = np.array([[0.0, 0.0], [1.2, 0.15], [0.35, 0.95]])
base_area = signed_area2(TRIANGLE)

sample_transforms = [
    {"name": "identity", "M": np.eye(3), "expected_class": "direct identity"},
    {"name": "translation", "M": translation((1.65, 0.55)), "expected_class": "direct translation"},
    {"name": "half-turn", "M": half_turn((0.65, 0.2)), "expected_class": "direct rotation"},
    {"name": "rotation", "M": rotation(math.radians(55), (0.4, 0.25)), "expected_class": "direct rotation"},
    {"name": "reflection", "M": reflection(math.radians(28), (0.2, 0.1)), "expected_class": "opposite reflection"},
    {"name": "glide reflection", "M": glide_reflection(0.0, (0.0, 0.0), 1.25), "expected_class": "opposite glide"},
]

classifier_rows = []
orientation_checks = []
image_sets = []
for item in sample_transforms:
    M = item["M"]
    image = apply_transform(M, TRIANGLE)
    image_sets.append(image)
    det = float(np.linalg.det(M[:2, :2]))
    area_ratio = signed_area2(image) / base_area
    distance_error = float(np.max(np.abs(pairwise_distances(image) - pairwise_distances(TRIANGLE))))
    fixed = fixed_set_type(M)
    sense = determinant_sense(M)
    classifier_rows.append({
        "name": item["name"],
        "expected_class": item["expected_class"],
        "determinant": round(det, 8),
        "sense": sense,
        "fixed_set_type": fixed,
        "signed_area_ratio": round(area_ratio, 8),
        "max_distance_error": f"{distance_error:.2e}",
    })
    orientation_checks.append({
        "name": item["name"],
        "determinant": det,
        "signed_area_ratio": area_ratio,
        "distance_error": distance_error,
        "orientation_matches_determinant": bool(abs(area_ratio - det) < 1e-10),
    })

write_csv(PATHS["classifier_table"], classifier_rows)
write_json(PATHS["orientation_checks"], {
    "chapter": CHAPTER_NO,
    "source_span": "printed pages 39-49; PDF pages 57-67",
    "checks": orientation_checks,
    "all_orientation_ratios_match_determinants": all(row["orientation_matches_determinant"] for row in orientation_checks),
    "max_distance_error": max(row["distance_error"] for row in orientation_checks),
})
write_json(PATHS["sample_data"], {
    row["name"]: np.round(item["M"], 10).tolist()
    for row, item in zip(classifier_rows, sample_transforms)
})

fig, axes = plt.subplots(2, 3, figsize=(12, 7.2), constrained_layout=True)
colors = ["#90a4ae", "#00acc1", "#7cb342", "#fb8c00", "#8e24aa", "#e53935"]
for ax, item, image, color in zip(axes.flat, sample_transforms, image_sets, colors):
    draw_polygon(ax, TRIANGLE, edge="#607d8b", face="#eceff1", alpha=0.65, label="original", lw=1.1)
    draw_polygon(ax, image, edge=color, face=color, alpha=0.23, label="image", lw=1.6)
    for p, q in zip(TRIANGLE, image):
        draw_segment(ax, p, q, color="#9e9e9e", linewidth=0.9, linestyle="--")
    det = np.linalg.det(item["M"][:2, :2])
    ax.set_title(f"{item['name']}\ndet={det:+.0f}, {determinant_sense(item['M'])}, fixed: {fixed_set_type(item['M'])}")
    set_equal_window(ax, np.vstack([TRIANGLE, image]), margin=0.35)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("A triangle classifies the sense of an isometry", y=1.02, fontsize=13)
fig.savefig(PATHS["classifier_figure"], format="svg", bbox_inches="tight")
plt.close(fig)

classifier_df = pd.DataFrame(classifier_rows)
display(classifier_df)
display_artifact(PATHS["classifier_figure"], width=940)


,name,expected_class,determinant,sense,fixed_set_type,signed_area_ratio,max_distance_error
0,identity,direct identity,1.0,direct,whole plane,1.0,0.00e+00
1,translation,direct translation,1.0,direct,none,1.0,2.22e-16
2,half-turn,direct rotation,1.0,direct,point,1.0,2.22e-16
3,rotation,direct rotation,1.0,direct,point,1.0,0.00e+00
4,reflection,opposite reflection,-1.0,opposite,line,-1.0,2.22e-16
5,glide reflection,opposite glide,-1.0,opposite,none,-1.0,2.22e-16


## 2. Reflection Products and Half-Turn Products

The source chapter repeatedly reduces isometries to reflections. The product is readable from the mirrors: intersecting mirrors give a rotation by twice the angle between them, parallel mirrors give a translation twice the spacing, and perpendicular mirrors give a half-turn. The same cancellation idea explains why two half-turns form a translation and why translations commute.


In [3]:
theta = math.radians(34)
d = 0.55
O1 = np.array([-0.95, -0.25])
O2 = np.array([0.7, 0.45])
lab_triangle = np.array([[0.55, 0.22], [1.15, 0.38], [0.78, 0.82]])

R0 = reflection(0.0)
Rtheta = reflection(theta)
intersecting_product = Rtheta @ R0
expected_rotation = rotation(2 * theta)
parallel_product = reflect_horizontal(d) @ reflect_horizontal(0.0)
expected_parallel_translation = translation((0.0, 2 * d))
halfturn_product = half_turn(O2) @ half_turn(O1)
expected_halfturn_translation = translation(2 * (O2 - O1))
perpendicular_product = reflection(math.pi / 2) @ reflection(0.0)
expected_halfturn_origin = half_turn((0.0, 0.0))

x1, y1, x2, y2, dsym, gsym = sp.symbols("x1 y1 x2 y2 d g", real=True)
S0 = sp.Matrix([[1, 0, 0], [0, -1, 0], [0, 0, 1]])
Sd = sp.Matrix([[1, 0, 0], [0, -1, 2 * dsym], [0, 0, 1]])
T2d = sp.Matrix([[1, 0, 0], [0, 1, 2 * dsym], [0, 0, 1]])
H1 = sp.Matrix([[-1, 0, 2 * x1], [0, -1, 2 * y1], [0, 0, 1]])
H2 = sp.Matrix([[-1, 0, 2 * x2], [0, -1, 2 * y2], [0, 0, 1]])
TH = sp.Matrix([[1, 0, 2 * (x2 - x1)], [0, 1, 2 * (y2 - y1)], [0, 0, 1]])
Gsym = sp.Matrix([[1, 0, gsym], [0, -1, 0], [0, 0, 1]])
Gsym2 = sp.Matrix([[1, 0, 2 * gsym], [0, 1, 0], [0, 0, 1]])

symbolic_checks = {
    "parallel_reflections_translate": bool(Sd * S0 == T2d),
    "two_half_turns_translate": bool(H2 * H1 == TH),
    "glide_square_is_translation": bool(Gsym * Gsym == Gsym2),
}

reflection_checks = {
    "chapter": CHAPTER_NO,
    "numeric_residuals": {
        "intersecting_reflections_rotation": float(np.max(np.abs(intersecting_product - expected_rotation))),
        "parallel_reflections_translation": float(np.max(np.abs(parallel_product - expected_parallel_translation))),
        "perpendicular_reflections_halfturn": float(np.max(np.abs(perpendicular_product - expected_halfturn_origin))),
        "two_halfturns_translation": float(np.max(np.abs(halfturn_product - expected_halfturn_translation))),
    },
    "symbolic_checks": symbolic_checks,
}
write_json(PATHS["reflection_checks"], reflection_checks)

group_rows = [
    {"first_factor": "reflection", "second_factor": "reflection", "condition": "mirrors intersect", "result": "rotation", "sense": "direct"},
    {"first_factor": "reflection", "second_factor": "reflection", "condition": "mirrors are parallel", "result": "translation", "sense": "direct"},
    {"first_factor": "reflection", "second_factor": "reflection", "condition": "mirrors are perpendicular", "result": "half-turn", "sense": "direct"},
    {"first_factor": "half-turn", "second_factor": "half-turn", "condition": "centers are distinct", "result": "translation", "sense": "direct"},
    {"first_factor": "translation", "second_factor": "translation", "condition": "any translation vectors", "result": "translation", "sense": "direct"},
    {"first_factor": "reflection", "second_factor": "translation", "condition": "translation along the mirror", "result": "glide reflection or reflection if zero shift", "sense": "opposite"},
    {"first_factor": "half-turn", "second_factor": "translation", "condition": "any nonzero translation", "result": "half-turn", "sense": "direct"},
]
write_csv(PATHS["group_table"], group_rows)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)

ax = axes[0]
image_rot = apply_transform(intersecting_product, lab_triangle)
draw_polygon(ax, lab_triangle, edge="#546e7a", face="#eceff1", alpha=0.8)
draw_polygon(ax, image_rot, edge="#00897b", face="#80cbc4", alpha=0.35)
xx = np.linspace(-1.4, 1.6, 2)
ax.plot(xx, 0 * xx, color="#263238", lw=1.5, label="mirror 1")
ax.plot(xx, math.tan(theta) * xx, color="#00897b", lw=1.5, label="mirror 2")
ax.add_patch(Arc((0, 0), 0.75, 0.75, angle=0, theta1=0, theta2=math.degrees(2 * theta), color="#f57c00", lw=1.8))
ax.text(0.38, 0.25, "2 theta", color="#e65100")
set_equal_window(ax, np.vstack([lab_triangle, image_rot, [[-1.4, -0.7], [1.6, 1.1]]]), margin=0.2)
ax.set_title("Intersecting mirrors -> rotation")

ax = axes[1]
par_triangle = np.array([[-0.9, -0.28], [-0.35, -0.15], [-0.65, -0.63]])
image_par = apply_transform(parallel_product, par_triangle)
draw_polygon(ax, par_triangle, edge="#546e7a", face="#eceff1", alpha=0.8)
draw_polygon(ax, image_par, edge="#3949ab", face="#c5cae9", alpha=0.45)
ax.axhline(0, color="#263238", lw=1.4)
ax.axhline(d, color="#3949ab", lw=1.4)
for p, q in zip(par_triangle, image_par):
    draw_segment(ax, p, q, color="#5c6bc0", linewidth=0.9, linestyle="--")
ax.annotate("2d", xy=(0.25, 1.1), xytext=(0.25, 0.0), arrowprops={"arrowstyle": "<->", "color": "#3949ab"}, ha="left", color="#283593")
set_equal_window(ax, np.vstack([par_triangle, image_par, [[-1.2, -0.8], [0.8, 1.25]]]), margin=0.2)
ax.set_title("Parallel mirrors -> translation")

ax = axes[2]
P = np.array([[-0.3, 0.95]])
P1 = apply_transform(half_turn(O1), P)
P2 = apply_transform(halfturn_product, P)
for c, label, color in [(O1, "O1", "#6d4c41"), (O2, "O2", "#ad1457")]:
    ax.add_patch(Circle(c, 0.055, color=color))
    ax.text(c[0] + 0.06, c[1] + 0.05, label, color=color)
ax.plot([P[0, 0], P1[0, 0], P2[0, 0]], [P[0, 1], P1[0, 1], P2[0, 1]], "o--", color="#546e7a")
for point, label in [(P[0], "P"), (P1[0], "H1(P)"), (P2[0], "H2H1(P)")]:
    ax.text(point[0] + 0.04, point[1] + 0.04, label, fontsize=8)
ax.annotate("2(O2-O1)", xy=P2[0], xytext=P[0], arrowprops={"arrowstyle": "->", "color": "#ad1457", "lw": 1.5}, color="#ad1457")
set_equal_window(ax, np.vstack([P, P1, P2, O1[None, :], O2[None, :]]), margin=0.35)
ax.set_title("Two half-turns -> translation")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("Products of elementary isometries", y=1.04, fontsize=13)
fig.savefig(PATHS["reflection_lab_figure"], format="svg", bbox_inches="tight")
plt.close(fig)

display(pd.DataFrame(group_rows))
display_artifact(PATHS["reflection_lab_figure"], width=980)


,first_factor,second_factor,condition,result,sense
0,reflection,reflection,mirrors intersect,rotation,direct
1,reflection,reflection,mirrors are parallel,translation,direct
2,reflection,reflection,mirrors are perpendicular,half-turn,direct
3,half-turn,half-turn,centers are distinct,translation,direct
4,translation,translation,any translation vectors,translation,direct
5,reflection,translation,translation along the mirror,glide reflection or reflection if zero shift,opposite
6,half-turn,translation,any nonzero translation,half-turn,direct


## 3. Glide Reflection: Opposite Sense with No Fixed Point

A glide reflection is easiest to see on a strip: reflect across the axis, then slide along the same axis. The map below is `G(x, y) = (x + g, -y)`. It reverses orientation like a reflection, but the slide prevents any point from staying fixed. Its visible trace is stronger than a formula: for every point `P`, the midpoint of `P` and `G(P)` lies on the axis.


In [4]:
g = 1.15
G = glide_reflection(0.0, (0.0, 0.0), g)
base_motif = np.array([
    [-0.33, 0.48],
    [-0.02, 0.78],
    [0.33, 0.63],
    [0.12, 0.51],
    [0.36, 0.39],
    [-0.08, 0.38],
])

fig, ax = plt.subplots(figsize=(10, 2.9), constrained_layout=True)
for n in range(-4, 5):
    Mn = matrix_power(G, n)
    pts = apply_transform(Mn, base_motif)
    color = "#26a69a" if n % 2 == 0 else "#ef5350"
    draw_polygon(ax, pts, edge=color, face=color, alpha=0.26, lw=1.2)
    ax.text(pts[:, 0].mean(), pts[:, 1].mean(), f"G^{n}", ha="center", va="center", fontsize=7, color="#263238")
ax.axhline(0, color="#263238", lw=1.5)
ax.text(-4.9, 0.08, "glide axis", color="#263238", fontsize=9)
probe = np.array([[-2.2, 0.65], [-0.2, 0.55], [1.8, 0.72]])
probe_image = apply_transform(G, probe)
midpoints = 0.5 * (probe + probe_image)
for p, q, m in zip(probe, probe_image, midpoints):
    draw_segment(ax, p, q, color="#5d4037", linewidth=1.0, linestyle="--")
    ax.plot(m[0], m[1], "o", color="#5d4037", ms=4)
ax.set_xlim(-5.1, 5.3)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect("equal", adjustable="box")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("A glide reflection alternates sides while sliding along the axis")
fig.savefig(PATHS["glide_svg"], format="svg", bbox_inches="tight")
plt.close(fig)

html_fig = go.Figure()
html_fig.add_trace(go.Scatter(x=[-5, 5.2], y=[0, 0], mode="lines", line={"color": "#263238", "width": 3}, name="glide axis"))
for n in range(-4, 5):
    pts = apply_transform(matrix_power(G, n), base_motif)
    pts_closed = np.vstack([pts, pts[0]])
    html_fig.add_trace(go.Scatter(
        x=pts_closed[:, 0],
        y=pts_closed[:, 1],
        mode="lines",
        fill="toself",
        fillcolor="rgba(38,166,154,0.22)" if n % 2 == 0 else "rgba(239,83,80,0.22)",
        line={"color": "#26a69a" if n % 2 == 0 else "#ef5350", "width": 2},
        name=f"G^{n} motif",
        hovertemplate=f"step G^{n}<br>orientation {'direct-side' if n % 2 == 0 else 'reflected-side'}<extra></extra>",
    ))
for idx, (p, q, m) in enumerate(zip(probe, probe_image, midpoints), start=1):
    html_fig.add_trace(go.Scatter(
        x=[p[0], m[0], q[0]],
        y=[p[1], m[1], q[1]],
        mode="lines+markers",
        line={"dash": "dot", "color": "#6d4c41"},
        marker={"size": [7, 8, 7], "color": ["#6d4c41", "#ffb300", "#6d4c41"]},
        name=f"midpoint trace {idx}",
        hovertemplate="P, midpoint, G(P)<extra></extra>",
    ))
html_fig.update_layout(
    title="Glide reflection midpoint traces",
    xaxis={"scaleanchor": "y", "range": [-5.1, 5.3], "zeroline": False},
    yaxis={"range": [-1.2, 1.2], "zeroline": False},
    width=920,
    height=360,
    margin={"l": 20, "r": 20, "t": 55, "b": 20},
    template="plotly_white",
)
html_fig.write_html(PATHS["glide_html"], include_plotlyjs=True, full_html=True)

rank_B = np.linalg.matrix_rank(G[:2, :2] - np.eye(2))
rank_aug = np.linalg.matrix_rank(np.c_[G[:2, :2] - np.eye(2), -G[:2, 2]])
glide_checks = {
    "chapter": CHAPTER_NO,
    "determinant": float(np.linalg.det(G[:2, :2])),
    "sense": determinant_sense(G),
    "fixed_set_type": fixed_set_type(G),
    "no_fixed_point_rank_test": {"rank_A_minus_I": int(rank_B), "rank_augmented": int(rank_aug)},
    "max_midpoint_axis_residual": float(np.max(np.abs(midpoints[:, 1]))),
    "square_residual_to_translation_2g": float(np.max(np.abs(G @ G - translation((2 * g, 0.0))))),
    "distance_error_on_probe": float(np.max(np.abs(pairwise_distances(apply_transform(G, probe)) - pairwise_distances(probe)))),
}
write_json(PATHS["glide_checks"], glide_checks)

display_artifact(PATHS["glide_svg"], width=940)
display_artifact(PATHS["glide_html"], width=940)


## 4. Hjelmslev Midpoint Scaffold

The chapter's midpoint theorem is a compact way to detect the hidden isometry between two lines. If every point on one line is paired by the same isometry with a point on another line, then the midpoints of the joining segments either form a line or collapse to one point. The two cases below isolate the alternatives: a reflection gives a visible midpoint line, while a half-turn makes every midpoint the center.


In [5]:
line_points = np.c_[np.linspace(-2.2, 2.2, 6), np.ones(6)]
hjelmslev_cases = [
    {"name": "reflection across x-axis", "M": reflection(0.0), "color": "#00838f"},
    {"name": "half-turn about origin", "M": half_turn((0.0, 0.0)), "color": "#c62828"},
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
hjelmslev_summary = []
for ax, case in zip(axes, hjelmslev_cases):
    image = apply_transform(case["M"], line_points)
    mids = 0.5 * (line_points + image)
    ax.plot(line_points[:, 0], line_points[:, 1], "o-", color="#546e7a", label="source line")
    ax.plot(image[:, 0], image[:, 1], "o-", color=case["color"], label="image line")
    for p, q, m in zip(line_points, image, mids):
        draw_segment(ax, p, q, color="#9e9e9e", linewidth=0.9, linestyle="--")
        ax.plot(m[0], m[1], "o", color="#f9a825", ms=5)
    centered = mids - mids.mean(axis=0)
    rank = int(np.linalg.matrix_rank(centered, tol=1e-10))
    spread = float(np.max(np.linalg.norm(centered, axis=1)))
    hjelmslev_summary.append({
        "case": case["name"],
        "midpoint_rank_after_centering": rank,
        "midpoint_spread": spread,
        "classification": "coincident" if spread < 1e-10 else "collinear",
    })
    ax.set_title(case["name"])
    set_equal_window(ax, np.vstack([line_points, image, mids]), margin=0.4)
    ax.legend(loc="upper right", fontsize=8)
fig.suptitle("Midpoints from a line to its isometric image", y=1.04, fontsize=13)
fig.savefig(PATHS["hjelmslev_figure"], format="svg", bbox_inches="tight")
plt.close(fig)

write_json(PATHS["hjelmslev_checks"], {
    "chapter": CHAPTER_NO,
    "cases": hjelmslev_summary,
    "all_cases_are_collinear_or_coincident": all(row["midpoint_rank_after_centering"] <= 1 for row in hjelmslev_summary),
})

display(pd.DataFrame(hjelmslev_summary))
display_artifact(PATHS["hjelmslev_figure"], width=900)


,case,midpoint_rank_after_centering,midpoint_spread,classification
0,reflection across x-axis,1,2.200000e+00,collinear
1,half-turn about origin,0,1.324665e-16,coincident


## 5. Strip Patterns as Generator Recipes

A strip pattern is not just a repeated picture; it is a repeated picture together with the isometries that preserve the whole strip. The gallery below uses one asymmetric motif so that the generators remain visible. Dashed vertical lines mark mirror axes, dotted horizontal lines mark a horizontal mirror, and filled dots mark half-turn centers.


In [6]:
motif = np.array([
    [-0.18, -0.22],
    [-0.18, 0.24],
    [0.20, 0.13],
    [0.04, 0.02],
    [0.25, -0.13],
])
period = 1.0
row_defs = [
    {"key": "S1", "name": "translation only", "generators": "one translation", "abstract_group": "C_infty", "orientation_generators": "+"},
    {"key": "S2", "name": "glide generated", "generators": "one glide reflection", "abstract_group": "C_infty", "orientation_generators": "-"},
    {"key": "S3", "name": "vertical mirrors", "generators": "two parallel reflections", "abstract_group": "D_infty", "orientation_generators": "-,-"},
    {"key": "S4", "name": "half-turn centers", "generators": "two half-turns", "abstract_group": "D_infty", "orientation_generators": "+,+"},
    {"key": "S5", "name": "vertical mirror plus half-turn", "generators": "one reflection and one half-turn", "abstract_group": "D_infty", "orientation_generators": "-,+"},
    {"key": "S6", "name": "horizontal mirror plus translation", "generators": "one translation and one reflection", "abstract_group": "C_infty x D_1", "orientation_generators": "+,-"},
    {"key": "S7", "name": "three reflection directions", "generators": "horizontal and vertical reflections", "abstract_group": "D_infty x D_1", "orientation_generators": "-,-"},
]


def draw_motif(ax, center, A=None, color="#00897b", alpha=0.28):
    A = np.eye(2) if A is None else np.asarray(A, dtype=float)
    pts = motif @ A.T + np.asarray(center)
    patch = Polygon(pts, closed=True, facecolor=color, edgecolor=color, lw=1.0, alpha=alpha)
    ax.add_patch(patch)
    closed = np.vstack([pts, pts[0]])
    ax.plot(closed[:, 0], closed[:, 1], color=color, lw=1.0)


fig, ax = plt.subplots(figsize=(11.5, 8.4), constrained_layout=True)
xs = range(-4, 5)
for idx, row in enumerate(row_defs):
    y0 = len(row_defs) - idx
    ax.text(-5.25, y0, f"{row['key']}  {row['name']}", ha="left", va="center", fontsize=9, color="#263238")
    ax.plot([-4.35, 4.55], [y0, y0], color="#eceff1", lw=1.0)
    if row["key"] == "S1":
        for n in xs:
            draw_motif(ax, (n * period, y0), np.eye(2), "#00897b")
    elif row["key"] == "S2":
        for n in xs:
            A = np.diag([1, 1 if n % 2 == 0 else -1])
            draw_motif(ax, (n * period * 0.62, y0 + (0.19 if n % 2 == 0 else -0.19)), A, "#d81b60")
        ax.plot([-4.35, 4.55], [y0, y0], color="#ad1457", lw=0.9, ls=":")
    elif row["key"] == "S3":
        for n in range(-3, 4):
            ax.plot([n + 0.5, n + 0.5], [y0 - 0.35, y0 + 0.35], color="#455a64", lw=0.9, ls="--")
            draw_motif(ax, (n + 0.18, y0 + 0.04), np.eye(2), "#5e35b1")
            draw_motif(ax, (n + 0.82, y0 + 0.04), np.diag([-1, 1]), "#5e35b1")
    elif row["key"] == "S4":
        for n in xs:
            A = np.eye(2) if n % 2 == 0 else -np.eye(2)
            draw_motif(ax, (n * 0.75, y0 + (0.17 if n % 2 == 0 else -0.17)), A, "#f57c00")
            if n % 2 == 0:
                ax.plot(n * 0.75 + 0.38, y0, "o", color="#ef6c00", ms=4)
    elif row["key"] == "S5":
        for n in range(-3, 4):
            ax.plot([n + 0.5, n + 0.5], [y0 - 0.35, y0 + 0.35], color="#455a64", lw=0.9, ls="--")
            ax.plot(n, y0, "o", color="#2e7d32", ms=4)
            draw_motif(ax, (n + 0.18, y0 + 0.18), np.eye(2), "#43a047")
            draw_motif(ax, (n + 0.82, y0 - 0.18), -np.diag([-1, 1]), "#43a047")
    elif row["key"] == "S6":
        ax.plot([-4.35, 4.55], [y0, y0], color="#455a64", lw=0.9, ls=":")
        for n in xs:
            draw_motif(ax, (n * period, y0 + 0.19), np.eye(2), "#039be5")
            draw_motif(ax, (n * period, y0 - 0.19), np.diag([1, -1]), "#039be5")
    elif row["key"] == "S7":
        ax.plot([-4.35, 4.55], [y0, y0], color="#455a64", lw=0.9, ls=":")
        for n in range(-3, 4):
            ax.plot([n + 0.5, n + 0.5], [y0 - 0.38, y0 + 0.38], color="#455a64", lw=0.9, ls="--")
            for sx, sy in [(1, 1), (-1, 1), (1, -1), (-1, -1)]:
                draw_motif(ax, (n + 0.5 + 0.22 * sx, y0 + 0.16 * sy), np.diag([sx, sy]), "#6d4c41", alpha=0.20)

ax.set_xlim(-5.35, 4.85)
ax.set_ylim(0.45, len(row_defs) + 0.55)
ax.set_aspect("equal", adjustable="box")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Seven strip-pattern generator recipes using one asymmetric motif", fontsize=13)
fig.savefig(PATHS["strip_gallery"], format="svg", bbox_inches="tight")
plt.close(fig)

strip_rows = []
for row in row_defs:
    strip_rows.append({
        "key": row["key"],
        "name": row["name"],
        "generators": row["generators"],
        "abstract_group": row["abstract_group"],
        "orientation_generators": row["orientation_generators"],
        "visible_markers": "translation period; mirror lines and half-turn dots where present",
    })
write_csv(PATHS["strip_table"], strip_rows)

orientation_symbol_to_det = {"+": 1, "-": -1}
strip_checks = {
    "chapter": CHAPTER_NO,
    "row_count": len(strip_rows),
    "all_rows_have_generators": all(bool(row["generators"]) for row in strip_rows),
    "generator_determinants": {
        row["key"]: [orientation_symbol_to_det[token] for token in row["orientation_generators"].split(",")]
        for row in row_defs
    },
    "period": period,
}
write_json(PATHS["strip_checks"], strip_checks)

display(pd.DataFrame(strip_rows))
display_artifact(PATHS["strip_gallery"], width=980)


,key,name,generators,abstract_group,orientation_generators,visible_markers
0,S1,translation only,one translation,C_infty,+,translation period; mirror lines and half-turn...
1,S2,glide generated,one glide reflection,C_infty,-,translation period; mirror lines and half-turn...
2,S3,vertical mirrors,two parallel reflections,D_infty,"-,-",translation period; mirror lines and half-turn...
3,S4,half-turn centers,two half-turns,D_infty,"+,+",translation period; mirror lines and half-turn...
4,S5,vertical mirror plus half-turn,one reflection and one half-turn,D_infty,"-,+",translation period; mirror lines and half-turn...
5,S6,horizontal mirror plus translation,one translation and one reflection,C_infty x D_1,"+,-",translation period; mirror lines and half-turn...
6,S7,three reflection directions,horizontal and vertical reflections,D_infty x D_1,"-,-",translation period; mirror lines and half-turn...


## Applied Lab: Classify a Proposed Isometry

Edit the matrix in `candidate` and rerun the cell. A true Euclidean isometry has an orthogonal linear part; the determinant and fixed-set rank then tell you which branch of the chapter's catalog it belongs to. This lab deliberately reports the residual instead of silently accepting a nearly-isometry.


In [7]:
candidate = glide_reflection(math.radians(18), (0.2, -0.1), 0.85)
A = candidate[:2, :2]
orthogonality_residual = float(np.max(np.abs(A.T @ A - np.eye(2))))
det = float(np.linalg.det(A))
lab_report = {
    "orthogonality_residual": orthogonality_residual,
    "determinant": det,
    "sense": determinant_sense(candidate),
    "fixed_set_type": fixed_set_type(candidate),
    "catalog_branch": "direct: rotation or translation" if det > 0 else "opposite: reflection or glide reflection",
}
assert orthogonality_residual < 1e-10
pd.DataFrame([lab_report])


,orthogonality_residual,determinant,sense,fixed_set_type,catalog_branch
0,4.440892e-16,-1.0,opposite,none,opposite: reflection or glide reflection


## Final Sanity Checks

The final cell checks both the files and the geometry. File checks keep the notebook honest about its generated artifacts; numerical and symbolic checks keep the classification claims honest.


In [8]:
visual_artifacts = [
    PATHS["classifier_figure"],
    PATHS["reflection_lab_figure"],
    PATHS["glide_svg"],
    PATHS["glide_html"],
    PATHS["hjelmslev_figure"],
    PATHS["strip_gallery"],
]
check_artifacts = [
    PATHS["orientation_checks"],
    PATHS["reflection_checks"],
    PATHS["glide_checks"],
    PATHS["hjelmslev_checks"],
    PATHS["strip_checks"],
]
table_artifacts = [PATHS["classifier_table"], PATHS["group_table"], PATHS["strip_table"]]
data_artifacts = [PATHS["sample_data"]]

manifest_rows = []
for category, paths in [
    ("visual", visual_artifacts),
    ("check", check_artifacts),
    ("table", table_artifacts),
    ("data", data_artifacts),
]:
    for path in paths:
        manifest_rows.append({"category": category, "path": book_rel(path), "bytes": path.stat().st_size})
write_csv(PATHS["manifest"], manifest_rows)

visual_summary = {
    "chapter": CHAPTER_NO,
    "title": "Isometry in the Euclidean Plane",
    "source_span": {"printed_pages": "39-49", "pdf_pages": "57-67"},
    "figures": [book_rel(path) for path in visual_artifacts if path.suffix.lower() != ".html"],
    "html": [book_rel(path) for path in visual_artifacts if path.suffix.lower() == ".html"],
    "checks": [book_rel(path) for path in check_artifacts],
    "tables": [book_rel(path) for path in table_artifacts + [PATHS["manifest"]]],
    "data": [book_rel(path) for path in data_artifacts],
}
write_json(PATHS["visual_summary"], visual_summary)

all_artifacts = visual_artifacts + check_artifacts + table_artifacts + data_artifacts + [PATHS["manifest"], PATHS["visual_summary"]]
assert_artifacts(all_artifacts, min_bytes=80)

orientation_payload = json.loads(PATHS["orientation_checks"].read_text(encoding="utf-8"))
reflection_payload = json.loads(PATHS["reflection_checks"].read_text(encoding="utf-8"))
glide_payload = json.loads(PATHS["glide_checks"].read_text(encoding="utf-8"))
hjelmslev_payload = json.loads(PATHS["hjelmslev_checks"].read_text(encoding="utf-8"))
strip_payload = json.loads(PATHS["strip_checks"].read_text(encoding="utf-8"))

assert orientation_payload["all_orientation_ratios_match_determinants"]
assert orientation_payload["max_distance_error"] < 1e-10
assert all(reflection_payload["symbolic_checks"].values())
assert max(reflection_payload["numeric_residuals"].values()) < 1e-10
assert glide_payload["determinant"] < 0
assert glide_payload["fixed_set_type"] == "none"
assert glide_payload["max_midpoint_axis_residual"] < 1e-10
assert glide_payload["square_residual_to_translation_2g"] < 1e-10
assert hjelmslev_payload["all_cases_are_collinear_or_coincident"]
assert strip_payload["row_count"] == 7

sanity_summary = {
    "chapter": CHAPTER_NO,
    "artifact_count": len(all_artifacts),
    "visual_count": len(visual_artifacts),
    "max_orientation_distance_error": orientation_payload["max_distance_error"],
    "max_reflection_product_residual": max(reflection_payload["numeric_residuals"].values()),
    "glide_midpoint_axis_residual": glide_payload["max_midpoint_axis_residual"],
    "strip_pattern_rows": strip_payload["row_count"],
    "passed": True,
}
write_json(PATHS["sanity_summary"], sanity_summary)
assert_artifacts([PATHS["sanity_summary"]], min_bytes=80)

pd.DataFrame([sanity_summary])


,chapter,artifact_count,visual_count,max_orientation_distance_error,max_reflection_product_residual,glide_midpoint_axis_residual,strip_pattern_rows,passed
0,3,17,6,2.220446e-16,4.440892e-16,0.0,7,True
